In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
"""
SpotFake - Weibo Dataset (Modernized, khớp cấu trúc dataset gốc)
=================================================================
Cấu trúc thư mục dataset:
  weibo/
  ├── nonrumor_images/     ← ảnh real (label=0)
  ├── rumor_images/        ← ảnh fake (label=1)
  ├── tweets/              ← text, mỗi file tên = image_id
  ├── train_id.pickle
  ├── test_id.pickle
  └── validate_id.pickle

Cài đặt:
  pip install torch torchvision transformers pillow scikit-learn tqdm
"""

# =============================================================================
# 1. IMPORTS
# =============================================================================
import os
import pickle
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms
from torchvision.models import vgg19, VGG19_Weights

from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =============================================================================
# 2. CONFIG — chỉnh DATA_ROOT cho đúng đường dẫn trên máy bạn
# =============================================================================
CONFIG = {
    # ---- Paths ----
    "data_root":       "/kaggle/input/datasets/phngphngtrn/weiboo1",               # thư mục gốc chứa dataset
    "nonrumor_images": "nonrumor_images",     # label = 0 (real)
    "rumor_images":    "rumor_images",        # label = 1 (fake)
    "tweets_dir":      "tweets",              # thư mục chứa text
    "train_ids":       "train_id.pickle",
    "test_ids":        "test_id.pickle",
    "val_ids":         "validate_id.pickle",

    # ---- BERT ----
    "bert_model":      "bert-base-chinese",   # Weibo = tiếng Trung
    "max_len":         512,

    # ---- Model hyperparams (giữ như paper) ----
    "text_hidden":     32,
    "vis_hidden":      32,
    "repr_size":       32,
    "dropout":         0.2,

    # ---- Training ----
    "batch_size":      8,
    "epochs":          50,
    "lr":              2e-5,
    "warmup_ratio":    0.1,

    # ---- Misc ----
    "num_classes":     2,
    "seed":            42,
    "device":          "cuda" if torch.cuda.is_available() else "cpu",
    "save_dir":        "checkpoints",
}

torch.manual_seed(CONFIG["seed"])
print(f"[INFO] Device: {CONFIG['device']}")


# =============================================================================
# 3. HELPER FUNCTIONS
# =============================================================================
def load_ids(pickle_path):
    """Load danh sách IDs từ file pickle"""
    with open(pickle_path, "rb") as f:
        ids = pickle.load(f)
    return list(ids)


def find_image(image_id, nonrumor_dir, rumor_dir):
    """
    Tìm ảnh trong cả 2 thư mục.
    Trả về (image_path, label): label=0 nếu nonrumor, label=1 nếu rumor
    """
    for ext in [".jpg", ".jpeg", ".png", ".gif", ""]:
        p = os.path.join(nonrumor_dir, str(image_id) + ext)
        if os.path.exists(p):
            return p, 0
        p = os.path.join(rumor_dir, str(image_id) + ext)
        if os.path.exists(p):
            return p, 1
    return None, -1


def read_tweet_text(tweet_dir, image_id):
    """Đọc text từ thư mục tweets/"""
    for ext in ["", ".txt", ".json"]:
        p = os.path.join(tweet_dir, str(image_id) + ext)
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                return f.read().strip()
    return ""  # fallback nếu không tìm thấy


# =============================================================================
# 4. DATASET
# =============================================================================
class WeiboDataset(Dataset):
    def __init__(self, ids, cfg, tokenizer, transform=None):
        self.tokenizer = tokenizer
        self.max_len   = cfg["max_len"]
        self.transform = transform or self._default_transform()

        nonrumor_dir = os.path.join(cfg["data_root"], cfg["nonrumor_images"])
        rumor_dir    = os.path.join(cfg["data_root"], cfg["rumor_images"])
        tweet_dir    = os.path.join(cfg["data_root"], cfg["tweets_dir"])

        self.samples = []
        missing = 0
        for id_ in ids:
            img_path, label = find_image(id_, nonrumor_dir, rumor_dir)
            if img_path is None:
                missing += 1
                continue
            text = read_tweet_text(tweet_dir, id_)
            self.samples.append((text, img_path, label))

        print(f"  Loaded {len(self.samples)} samples | Skipped (no image): {missing}")

    @staticmethod
    def _default_transform():
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, img_path, label = self.samples[idx]

        # --- Text ---
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)

        # --- Image ---
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=0)
        image = self.transform(image)

        return input_ids, attn_mask, image, torch.tensor(label, dtype=torch.long)


# =============================================================================
# 5. MODEL
# =============================================================================
class SpotFake(nn.Module):
    """
    Text branch  : BERT pooler → FC(32) → ReLU → Dropout → FC(32) → ReLU
    Image branch : VGG19 features → Flatten → FC(32) → ReLU → Dropout → FC(32) → ReLU
    Fusion       : Concat(64) → FC(32) → ReLU → Dropout → FC(2)
    """
    def __init__(self, cfg):
        super().__init__()

        # Text
        self.bert    = BertModel.from_pretrained(cfg["bert_model"])
        bert_dim     = self.bert.config.hidden_size  # 768
        self.text_fc = nn.Sequential(
            nn.Linear(bert_dim, cfg["text_hidden"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["text_hidden"], cfg["repr_size"]),
            nn.ReLU(),
        )

        # Image
        vgg           = vgg19(weights=VGG19_Weights.DEFAULT)
        self.vgg_feat = vgg.features
        self.vgg_pool = vgg.avgpool
        vgg_flat_dim  = 512 * 7 * 7  # 25088
        self.image_fc = nn.Sequential(
            nn.Linear(vgg_flat_dim, cfg["vis_hidden"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["vis_hidden"], cfg["repr_size"]),
            nn.ReLU(),
        )

        # Fusion
        self.classifier = nn.Sequential(
            nn.Linear(cfg["repr_size"] * 2, cfg["repr_size"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["repr_size"], cfg["num_classes"]),
        )

    def forward(self, input_ids, attn_mask, images):
        text_repr  = self.text_fc(self.bert(input_ids=input_ids,
                                            attention_mask=attn_mask).pooler_output)
        vgg_out    = self.vgg_pool(self.vgg_feat(images)).flatten(start_dim=1)
        image_repr = self.image_fc(vgg_out)
        logits     = self.classifier(torch.cat([text_repr, image_repr], dim=1))
        return logits


# =============================================================================
# 6. TRAIN / EVAL
# =============================================================================
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, attn_mask, images, labels in tqdm(loader, desc="  Train", leave=False):
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        images    = images.to(device)
        labels    = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.detach().argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, attn_mask, images, labels in tqdm(loader, desc="  Eval ", leave=False):
        logits = model(input_ids.to(device), attn_mask.to(device), images.to(device))
        total_loss += criterion(logits, labels.to(device)).item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average="weighted"),
        precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        recall_score(all_labels, all_preds, average="weighted", zero_division=0),
    )


# =============================================================================
# 7. MAIN
# =============================================================================
def main():
    cfg    = CONFIG
    device = cfg["device"]
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # Load IDs từ pickle
    train_ids = load_ids(os.path.join(cfg["data_root"], cfg["train_ids"]))
    test_ids  = load_ids(os.path.join(cfg["data_root"], cfg["test_ids"]))
    val_ids   = load_ids(os.path.join(cfg["data_root"], cfg["val_ids"]))
    print(f"[INFO] IDs → train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")

    # Tokenizer & Datasets
    print("[INFO] Loading bert-base-chinese tokenizer...")
    tokenizer = BertTokenizer.from_pretrained(cfg["bert_model"])

    print("[INFO] Building datasets...")
    train_ds = WeiboDataset(train_ids, cfg, tokenizer)
    val_ds   = WeiboDataset(val_ids,   cfg, tokenizer)
    test_ds  = WeiboDataset(test_ids,  cfg, tokenizer)

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)

    # Model
    print("[INFO] Building SpotFake model...")
    model = SpotFake(cfg).to(device)

    # Optimizer & Scheduler
    optimizer   = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-2)
    total_steps = len(train_loader) * cfg["epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(cfg["warmup_ratio"] * total_steps),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    # Training loop
    best_val_acc = 0.0
    best_ckpt    = os.path.join(cfg["save_dir"], "spotfake_weibo_best.pth")

    print(f"\n{'='*65}")
    print(f"  SpotFake Weibo | {cfg['epochs']} epochs | {device}")
    print(f"{'='*65}")

    for epoch in range(1, cfg["epochs"] + 1):
        print(f"\nEpoch {epoch:02d}/{cfg['epochs']}")
        tr_loss, tr_acc                        = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        vl_loss, vl_acc, vl_f1, vl_p, vl_r    = eval_epoch(model, val_loader, criterion, device)

        print(f"  Train → Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}")
        print(f"  Val   → Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  "
              f"F1: {vl_f1:.4f}  P: {vl_p:.4f}  R: {vl_r:.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_ckpt)
            print(f"  ✓ Best model saved (val_acc={best_val_acc:.4f})")

    # Final test
    print(f"\n{'='*65}")
    print("  Final TEST evaluation (best checkpoint)")
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    ts_loss, ts_acc, ts_f1, ts_p, ts_r = eval_epoch(model, test_loader, criterion, device)
    print(f"  Test → Loss: {ts_loss:.4f}  Acc: {ts_acc:.4f}  "
          f"F1: {ts_f1:.4f}  P: {ts_p:.4f}  R: {ts_r:.4f}")
    print(f"{'='*65}")


# =============================================================================
# 8. INFERENCE — dự đoán 1 sample
# =============================================================================
def predict(text, image_path, model_path="checkpoints/spotfake_weibo_best.pth"):
    cfg       = CONFIG
    device    = cfg["device"]
    tokenizer = BertTokenizer.from_pretrained(cfg["bert_model"])
    model     = SpotFake(cfg)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device).eval()

    enc       = tokenizer(text, max_length=cfg["max_len"], padding="max_length",
                          truncation=True, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)
    attn_mask = enc["attention_mask"].to(device)

    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image = tf(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_ids, attn_mask, image)
        probs  = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred   = logits.argmax(dim=1).item()

    label = "FAKE (Rumor)" if pred == 1 else "REAL (Non-rumor)"
    print(f"→ {label}  |  Real: {probs[0]:.3f}  Fake: {probs[1]:.3f}")
    return pred, probs


if __name__ == "__main__":
    main()

[INFO] Device: cuda
[INFO] IDs → train: 5415 | val: 843 | test: 1465
[INFO] Loading bert-base-chinese tokenizer...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[INFO] Building datasets...
  Loaded 0 samples | Skipped (no image): 5415
  Loaded 0 samples | Skipped (no image): 843
  Loaded 0 samples | Skipped (no image): 1465


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [7]:
import os, pickle

data_root = "/kaggle/input/datasets/phngphngtrn/weiboo1"  # chỉnh lại nếu khác

# Xem thử 5 ID đầu trong pickle
with open(f"{data_root}/train_id.pickle", "rb") as f:
    ids = list(pickle.load(f))
print("Sample IDs:", ids[:5])
print("Type:", type(ids[0]))

# Xem thử tên file ảnh trong 2 thư mục
nonrumor = os.listdir(f"{data_root}/nonrumor_images")[:5]
rumor    = os.listdir(f"{data_root}/rumor_images")[:5]
print("\nNonrumor files:", nonrumor)
print("Rumor files:   ", rumor)

# Xem thử tweets/
tweets = os.listdir(f"{data_root}/tweets")[:5]
print("\nTweets files:", tweets)

Sample IDs: ['3874351616957777', '3924019927281845', '3803929878271778', '3834862396354088', '3842718340477479']
Type: <class 'str'>

Nonrumor files: ['617fc8a9jw1evchnjnabpj20hs0dcwi3.jpg', '470bf257jw1etqpi9ucwhj20cg0hldkw.jpg', '70e11e0fjw1ey13pu3pj6j20h70ac75o.jpg', '684ebae3jw1ezdcaatzomj20c80c8q4z.jpg', '6a5ce645jw1eyz10bnbasj20c8257tgh.jpg']
Rumor files:    ['b9d012c5jw1eov2af7u3qj20ch07fjrk.jpg', '82e8a5b3jw1ee1hb2scmbj20h60qon0t.jpg', '97e9fc13jw1eo6ue3v0vij20cb07pdhk.jpg', '6766a160jw1e2p0z3ixz7j.jpg', 'aa4e9201jw1e62ffyepbsj20fa0p0dj4.jpg']

Tweets files: ['train_rumor.txt', 'test_nonrumor.txt', 'train_nonrumor.txt', 'test_rumor.txt']


In [9]:
"""
SpotFake - Weibo Dataset (Modernized, format 3-dòng đúng)
==========================================================
Format mỗi post trong txt file (3 dòng liên tiếp):
  Dòng 1: post_id|username|url|...|platform
  Dòng 2: img_url1|img_url2|...|null
  Dòng 3: nội dung text

Ảnh local: tên file = phần cuối URL (vd: a71ac854gw1dytin2zmk9j.jpg)
  - nonrumor_images/  → label = 0
  - rumor_images/     → label = 1

Cài đặt:
  pip install torch torchvision transformers pillow scikit-learn tqdm
"""

# =============================================================================
# 1. IMPORTS
# =============================================================================
import os
import pickle
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torchvision import transforms
from torchvision.models import vgg19, VGG19_Weights

from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# =============================================================================
# 2. CONFIG
# =============================================================================
CONFIG = {
    "data_root":       "/kaggle/input/datasets/phngphngtrn/weiboo1",
    "nonrumor_images": "nonrumor_images",
    "rumor_images":    "rumor_images",
    "tweets_dir":      "tweets",
    "train_ids":       "train_id.pickle",
    "test_ids":        "test_id.pickle",
    "val_ids":         "validate_id.pickle",

    "bert_model":      "bert-base-chinese",
    "max_len":         512,

    "text_hidden":     32,
    "vis_hidden":      32,
    "repr_size":       32,
    "dropout":         0.2,

    "batch_size":      8,
    "epochs":          50,
    "lr":              2e-5,
    "warmup_ratio":    0.1,

    "num_classes":     2,
    "seed":            42,
    "device":          "cuda" if torch.cuda.is_available() else "cpu",
    "save_dir":        "checkpoints",
}

torch.manual_seed(CONFIG["seed"])
print(f"[INFO] Device: {CONFIG['device']}")


# =============================================================================
# 3. PARSE TXT FILES → dict: post_id → {text, images, label}
# =============================================================================
def parse_weibo_txt(filepath, label):
    """
    Đọc file txt format 3 dòng / post:
      Dòng 1: post_id|...metadata...
      Dòng 2: img_url1|img_url2|...|null
      Dòng 3: text content
    Trả về dict {post_id: {"text": str, "img_files": [str], "label": int}}
    """
    posts = {}
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [l.rstrip("\n") for l in f.readlines()]

    i = 0
    while i + 2 < len(lines):
        meta_line  = lines[i].strip()
        img_line   = lines[i + 1].strip()
        text_line  = lines[i + 2].strip()
        i += 3

        # Bỏ qua block rỗng
        if not meta_line:
            continue

        # post_id = field đầu tiên
        post_id = meta_line.split("|")[0].strip()
        if not post_id:
            continue

        # Lấy tên file ảnh từ URL (phần sau dấu / cuối)
        img_files = []
        for url in img_line.split("|"):
            url = url.strip()
            if url and url.lower() != "null" and url.startswith("http"):
                fname = url.split("/")[-1]
                if fname:
                    img_files.append(fname)

        posts[post_id] = {
            "text":      text_line,
            "img_files": img_files,
            "label":     label,
        }

    return posts


def build_post_db(cfg):
    """Gộp cả 4 file txt thành 1 dict lớn"""
    tweet_dir = os.path.join(cfg["data_root"], cfg["tweets_dir"])
    db = {}

    file_label_map = {
        "train_rumor.txt":    1,
        "test_rumor.txt":     1,
        "train_nonrumor.txt": 0,
        "test_nonrumor.txt":  0,
    }

    for fname, label in file_label_map.items():
        fpath = os.path.join(tweet_dir, fname)
        if not os.path.exists(fpath):
            print(f"  [WARN] Không tìm thấy: {fpath}")
            continue
        parsed = parse_weibo_txt(fpath, label)
        db.update(parsed)
        print(f"  Parsed {fname}: {len(parsed)} posts")

    print(f"  Total posts in DB: {len(db)}")
    return db


def load_ids(pickle_path):
    with open(pickle_path, "rb") as f:
        return list(pickle.load(f))


# =============================================================================
# 4. DATASET
# =============================================================================
class WeiboDataset(Dataset):
    def __init__(self, ids, post_db, cfg, tokenizer, transform=None):
        self.tokenizer = tokenizer
        self.max_len   = cfg["max_len"]
        self.transform = transform or self._default_transform()

        nonrumor_dir = os.path.join(cfg["data_root"], cfg["nonrumor_images"])
        rumor_dir    = os.path.join(cfg["data_root"], cfg["rumor_images"])

        self.samples = []
        missing_id   = 0
        missing_img  = 0

        for post_id in ids:
            if post_id not in post_db:
                missing_id += 1
                continue

            info      = post_db[post_id]
            label     = info["label"]
            text      = info["text"]
            img_dir   = rumor_dir if label == 1 else nonrumor_dir

            # Tìm ảnh đầu tiên tồn tại
            img_path = None
            for fname in info["img_files"]:
                candidate = os.path.join(img_dir, fname)
                if os.path.exists(candidate):
                    img_path = candidate
                    break

            if img_path is None:
                missing_img += 1
                continue

            self.samples.append((text, img_path, label))

        total = len(self.samples)
        print(f"  Loaded {total} | Missing ID: {missing_id} | Missing img: {missing_img}")

    @staticmethod
    def _default_transform():
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        text, img_path, label = self.samples[idx]

        # Text
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].squeeze(0)
        attn_mask = enc["attention_mask"].squeeze(0)

        # Image
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224), color=0)
        image = self.transform(image)

        return input_ids, attn_mask, image, torch.tensor(label, dtype=torch.long)


# =============================================================================
# 5. MODEL
# =============================================================================
class SpotFake(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Text branch
        self.bert    = BertModel.from_pretrained(cfg["bert_model"])
        bert_dim     = self.bert.config.hidden_size  # 768
        self.text_fc = nn.Sequential(
            nn.Linear(bert_dim, cfg["text_hidden"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["text_hidden"], cfg["repr_size"]),
            nn.ReLU(),
        )

        # Image branch
        vgg           = vgg19(weights=VGG19_Weights.DEFAULT)
        self.vgg_feat = vgg.features
        self.vgg_pool = vgg.avgpool
        self.image_fc = nn.Sequential(
            nn.Linear(512 * 7 * 7, cfg["vis_hidden"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["vis_hidden"], cfg["repr_size"]),
            nn.ReLU(),
        )

        # Fusion
        self.classifier = nn.Sequential(
            nn.Linear(cfg["repr_size"] * 2, cfg["repr_size"]),
            nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(cfg["repr_size"], cfg["num_classes"]),
        )

    def forward(self, input_ids, attn_mask, images):
        text_repr  = self.text_fc(
            self.bert(input_ids=input_ids, attention_mask=attn_mask).pooler_output
        )
        image_repr = self.image_fc(
            self.vgg_pool(self.vgg_feat(images)).flatten(start_dim=1)
        )
        return self.classifier(torch.cat([text_repr, image_repr], dim=1))


# =============================================================================
# 6. TRAIN / EVAL
# =============================================================================
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, attn_mask, images, labels in tqdm(loader, desc="  Train", leave=False):
        input_ids = input_ids.to(device)
        attn_mask = attn_mask.to(device)
        images    = images.to(device)
        labels    = labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn_mask, images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        all_preds.extend(logits.detach().argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), accuracy_score(all_labels, all_preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []

    for input_ids, attn_mask, images, labels in tqdm(loader, desc="  Eval ", leave=False):
        logits = model(input_ids.to(device), attn_mask.to(device), images.to(device))
        total_loss += criterion(logits, labels.to(device)).item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

    return (
        total_loss / len(loader),
        accuracy_score(all_labels, all_preds),
        f1_score(all_labels, all_preds, average="weighted"),
        precision_score(all_labels, all_preds, average="weighted", zero_division=0),
        recall_score(all_labels, all_preds, average="weighted", zero_division=0),
    )


# =============================================================================
# 7. MAIN
# =============================================================================
def main():
    cfg    = CONFIG
    device = cfg["device"]
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # Build post database từ 4 txt files
    print("[INFO] Parsing tweet files...")
    post_db = build_post_db(cfg)

    # Load split IDs
    train_ids = load_ids(os.path.join(cfg["data_root"], cfg["train_ids"]))
    val_ids   = load_ids(os.path.join(cfg["data_root"], cfg["val_ids"]))
    test_ids  = load_ids(os.path.join(cfg["data_root"], cfg["test_ids"]))
    print(f"[INFO] IDs → train: {len(train_ids)} | val: {len(val_ids)} | test: {len(test_ids)}")

    # Tokenizer
    print("[INFO] Loading bert-base-chinese tokenizer...")
    tokenizer = BertTokenizer.from_pretrained(cfg["bert_model"])

    # Datasets
    print("[INFO] Building datasets...")
    train_ds = WeiboDataset(train_ids, post_db, cfg, tokenizer)
    val_ds   = WeiboDataset(val_ids,   post_db, cfg, tokenizer)
    test_ds  = WeiboDataset(test_ids,  post_db, cfg, tokenizer)

    assert len(train_ds) > 0, "Train dataset rỗng! Kiểm tra lại đường dẫn data_root."

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=cfg["batch_size"],
                              shuffle=False, num_workers=2, pin_memory=True)

    # Model
    print("[INFO] Building SpotFake model...")
    model = SpotFake(cfg).to(device)

    # Optimizer & Scheduler
    optimizer   = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-2)
    total_steps = len(train_loader) * cfg["epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(cfg["warmup_ratio"] * total_steps),
        num_training_steps=total_steps,
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    best_ckpt    = os.path.join(cfg["save_dir"], "spotfake_weibo_best.pth")

    print(f"\n{'='*65}")
    print(f"  SpotFake Weibo | {cfg['epochs']} epochs | {device}")
    print(f"{'='*65}")

    for epoch in range(1, cfg["epochs"] + 1):
        print(f"\nEpoch {epoch:02d}/{cfg['epochs']}")
        tr_loss, tr_acc                     = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        vl_loss, vl_acc, vl_f1, vl_p, vl_r = eval_epoch(model, val_loader, criterion, device)

        print(f"  Train → Loss: {tr_loss:.4f}  Acc: {tr_acc:.4f}")
        print(f"  Val   → Loss: {vl_loss:.4f}  Acc: {vl_acc:.4f}  "
              f"F1: {vl_f1:.4f}  P: {vl_p:.4f}  R: {vl_r:.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            torch.save(model.state_dict(), best_ckpt)
            print(f"  ✓ Best model saved (val_acc={best_val_acc:.4f})")

    # Final test
    print(f"\n{'='*65}")
    print("  Final TEST evaluation (best checkpoint)")
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    ts_loss, ts_acc, ts_f1, ts_p, ts_r = eval_epoch(model, test_loader, criterion, device)
    print(f"  Test → Loss: {ts_loss:.4f}  Acc: {ts_acc:.4f}  "
          f"F1: {ts_f1:.4f}  P: {ts_p:.4f}  R: {ts_r:.4f}")
    print(f"{'='*65}")


if __name__ == "__main__":
    main()

[INFO] Device: cuda
[INFO] Parsing tweet files...
  Parsed train_rumor.txt: 3748 posts
  Parsed test_rumor.txt: 1000 posts
  Parsed train_nonrumor.txt: 3783 posts
  Parsed test_nonrumor.txt: 996 posts
  Total posts in DB: 9527
[INFO] IDs → train: 5415 | val: 843 | test: 1465
[INFO] Loading bert-base-chinese tokenizer...
[INFO] Building datasets...
  Loaded 5415 | Missing ID: 0 | Missing img: 0
  Loaded 843 | Missing ID: 0 | Missing img: 0
  Loaded 1465 | Missing ID: 0 | Missing img: 0
[INFO] Building SpotFake model...


config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  Train → Loss: 0.0010  Acc: 0.9998
  Val   → Loss: 1.5619  Acc: 0.8873  F1: 0.8873  P: 0.8967  R: 0.8873

Epoch 48/50


  Train → Loss: 0.0009  Acc: 0.9998
  Val   → Loss: 1.3525  Acc: 0.9075  F1: 0.9076  P: 0.9100  R: 0.9075

Epoch 49/50


  Train → Loss: 0.0009  Acc: 0.9998
  Val   → Loss: 1.2511  Acc: 0.9075  F1: 0.9076  P: 0.9095  R: 0.9075

Epoch 50/50


  Train → Loss: 0.0000  Acc: 1.0000
  Val   → Loss: 1.2576  Acc: 0.9075  F1: 0.9076  P: 0.9095  R: 0.9075

  Final TEST evaluation (best checkpoint)


  Test → Loss: 0.4604  Acc: 0.9481  F1: 0.9481  P: 0.9485  R: 0.9481


In [11]:
import requests
from io import BytesIO

# 1. Điền câu text 
# Ví dụ: "Sốc! Thường xuyên ăn tỏi sẽ bị ngốc!"
text_cua_ban = "数据流" 

# 2. Điền link một bức ảnh 
url_anh = "https://image2url.com/r2/default/images/1774755170316-f4d4b7c7-d14b-4f60-bad8-d002b7edafc6.png"

# Tải ảnh về Kaggle tạm thời
response = requests.get(url_anh)
img_path_test = "anh_test_thuc_te.jpg"
with open(img_path_test, "wb") as f:
    f.write(response.content)

print(f"📄 TEXT CỦA BẠN: {text_cua_ban}")
print("-" * 30)

# 3. Cho mô hình phán quyết!
predict(text=text_cua_ban, image_path=img_path_test, model_path="checkpoints/spotfake_weibo_best.pth")

📄 TEXT CỦA BẠN: 数据流
------------------------------


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


→ REAL (Non-rumor)  |  Real: 1.000  Fake: 0.000


(0, array([9.9977154e-01, 2.2852128e-04], dtype=float32))